In [33]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

In [34]:
load_dotenv(dotenv_path='../.env')

True

In [35]:
model = ChatOpenAI(model='gpt-4o-mini')

In [36]:
class BlogState(TypedDict):
    title: str
    outline: str
    content: str
    eval_score: int

In [37]:
def create_outline(state: BlogState) -> BlogState:
    # fetch title from state
    title = state['title']

    # create outline using LLM
    prompt = f"Create a detailed outline for a blog post with the title: {title}"
    outline = model.invoke(prompt)

    # update state with outline
    state['outline'] = outline
    return state


In [38]:
def create_blog(state: BlogState) -> BlogState:
    title = state['title']
    outline = state['outline']

    prompt = f"Write a blog post based on the following title and outline:\n\nTitle: {title}\n\nOutline: {outline}"

    blog = model.invoke(prompt).content

    state['content'] = blog
    return state

In [39]:
def evaluate_blog(state: BlogState) -> BlogState:
    content = state['content']
    outline = state['outline']

    prompt = f"Evaluate the following blog content based on the provided outline and generate a score out of 10(1 means poor, 10 means excellent). Provide feedback on how well the content follows the outline and suggest improvements.\n\nOutline: {outline}\n\nContent: {content}"

    score = model.invoke(prompt).content
    state['eval_score'] = score
    return state

In [40]:
graph = StateGraph(BlogState)

graph.add_node('create_outline', create_outline)
graph.add_node('create_blog', create_blog)
graph.add_node('evaluate_blog', evaluate_blog)

graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', 'evaluate_blog')
graph.add_edge('evaluate_blog', END)

workflow = graph.compile()

In [41]:
initial_state = {'title': 'Samay Raina controversy'}

final_state = workflow.invoke(initial_state)
print("Final Blog Content:", final_state['content'])
print("Evaluation Score:", final_state['eval_score'])

Final Blog Content: # Samay Raina Controversy: A Deep Dive into the Recent Fray

## I. Introduction

### A. Brief Overview of Samay Raina

Samay Raina is a prominent figure in the Indian comedy scene, known for his sharp wit and relatable humor. As a leading comedian and YouTuber, he has captivated audiences across the country, using his platform to address various social issues while also entertaining millions. His rise to fame is marked by a unique blend of stand-up comedy, sketches, and engaging content, solidifying his position as a voice of modern Indian comedy.

### B. Introduction to the Controversy

However, not all news surrounding Samay Raina has been positive. A recent controversy has sparked significant debate, drawing attention to the cultural sensitivities of comedy in India today. The need for discussion is critical as it reflects the complex interplay of humor, societal norms, and the reactions they can provoke.

## II. Background Information

### A. Samay Raina's Caree

In [42]:
print(final_state['eval_score'])

### Evaluation Score: 8/10

### Feedback

The blog content largely adheres to the provided outline, effectively covering the necessary points and creating a comprehensive narrative around the Samay Raina controversy. Here’s a breakdown of how well the content follows the outline:

**Strengths:**

1. **Structured Format**: The blog is organized clearly according to the outline, with distinct sections that logically flow from one to the next.
  
2. **Comprehensive Coverage**: Each section is addressed adequately, providing insights into Samay Raina's background, the controversy itself, different perspectives, and the broader implications on comedy. 

3. **Engaging Introduction**: The introduction does a good job of synopsizing Samay Raina’s career and setting the stage for the controversy, making it clear why this discussion is relevant.

4. **Balanced Perspectives**: Both supporters and critics of Raina's actions are addressed, which adds depth and caters to a broader audience.

5. **Ca